# 05 — GradCAM Visual Analysis

**ICS555 Capstone: African Landmark Recognition**

Generates GradCAM heatmaps comparing CNN-from-scratch (A0) vs ResNet-50 (A4) to understand
what each model uses to make its predictions.

Paper figure: 3 rows × 2 columns
1. Case where A4 succeeds but A0 fails
2. Case where both fail (hardest class)
3. Clean success for A4

In [ ]:
import os
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    !git clone https://github.com/zamsi-ajegetina/landmark-recognition.git
    %cd landmark-recognition
    !pip install -q -r requirements.txt
    from google.colab import drive
    drive.mount('/content/drive')
    !ln -sf /content/drive/MyDrive/landmark_images landmark_images
    # Copy checkpoints from Drive
    !cp /content/drive/MyDrive/checkpoints/*.pt checkpoints/ 2>/dev/null || true

In [ ]:
%matplotlib inline
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from torchvision import transforms

sys.path.insert(0, '..')
from src.model import MyModel
from src.transfer import get_model_transfer_learning
from src.helpers import compute_mean_and_std, get_data_location
from torchvision.datasets import ImageFolder

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 1. Load models

In [ ]:
# A0: CNN from scratch
model_a0 = MyModel(num_classes=50)
model_a0.load_state_dict(torch.load('checkpoints/A0_scratch.pt', map_location='cpu'))
model_a0.eval()

# A4: ResNet-50 full fine-tune
model_a4 = get_model_transfer_learning('resnet50', n_classes=50, finetune_strategy='full')
model_a4.load_state_dict(torch.load('checkpoints/A4_resnet50_full.pt', map_location='cpu'))
model_a4.eval()

print('Both models loaded.')

## 2. Build GradCAM extractor

In [ ]:
from torchcam.methods import GradCAM
from torchcam.utils import overlay_mask
from torchvision.transforms.functional import to_pil_image

mean, std = compute_mean_and_std()
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

def get_gradcam(model, target_layer, pil_img, class_idx=None):
    """Returns the GradCAM heatmap overlaid on the input image."""
    extractor = GradCAM(model, target_layer=target_layer)
    tensor = transform(pil_img).unsqueeze(0)
    logits = model(tensor)
    if class_idx is None:
        class_idx = logits.argmax(dim=1).item()
    activation_map = extractor(class_idx, logits)[0]
    extractor.remove_hooks()
    heatmap = to_pil_image(activation_map, mode='F')
    result  = overlay_mask(pil_img.resize((224, 224)), heatmap, alpha=0.5)
    return result, class_idx

# Target layers
# A0: last conv block — conv6 is the final Conv2d in MyModel
# A4: model.layer4[-1].conv2

print('GradCAM helpers ready.')

## 3. Find interesting examples

In [ ]:
data_path = Path(get_data_location())
test_ds   = ImageFolder(data_path / 'test')
class_names = test_ds.classes

def predict(model, pil_img):
    tensor = transform(pil_img).unsqueeze(0)
    with torch.no_grad():
        logits = model(tensor)
    return logits.argmax(dim=1).item()

# Find cases:
# - A4 correct, A0 wrong  → A4 advantage
# - Both wrong            → hard case
# - Both correct          → clean success
cases = {'a4_wins': [], 'both_fail': [], 'both_correct': []}

for img_path, true_label in test_ds.samples:
    if all(len(v) >= 3 for v in cases.values()):
        break
    img = Image.open(img_path).convert('RGB')
    pred_a0 = predict(model_a0, img)
    pred_a4 = predict(model_a4, img)

    if pred_a4 == true_label and pred_a0 != true_label and len(cases['a4_wins']) < 3:
        cases['a4_wins'].append((img_path, true_label))
    elif pred_a4 != true_label and pred_a0 != true_label and len(cases['both_fail']) < 3:
        cases['both_fail'].append((img_path, true_label))
    elif pred_a4 == true_label and pred_a0 == true_label and len(cases['both_correct']) < 3:
        cases['both_correct'].append((img_path, true_label))

for k, v in cases.items():
    print(f'{k}: {len(v)} examples found')

## 4. Paper figure: GradCAM comparison grid

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

row_titles = [
    'A4 succeeds, A0 fails',
    'Both fail (hardest class)',
    'Clean success (both correct)',
]
col_titles = ['Original', 'True label', 'A0 GradCAM', 'A4 GradCAM']

for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=11, fontweight='bold')

case_keys = ['a4_wins', 'both_fail', 'both_correct']

for row, (key, row_title) in enumerate(zip(case_keys, row_titles)):
    if not cases[key]:
        for col in range(4):
            axes[row, col].text(0.5, 0.5, 'No example found', ha='center', va='center')
            axes[row, col].axis('off')
        continue

    img_path, true_label = cases[key][0]
    img = Image.open(img_path).convert('RGB')
    label_str = class_names[true_label]

    # Col 0: original
    axes[row, 0].imshow(img.resize((224, 224)))
    axes[row, 0].set_ylabel(row_title, fontsize=9, rotation=0, labelpad=110, va='center')

    # Col 1: label
    axes[row, 1].imshow(img.resize((224, 224)))
    axes[row, 1].set_title(f'True: {label_str}', fontsize=8)

    # Col 2: A0 GradCAM
    try:
        # MyModel's final conv layer is named 'conv6'
        cam_a0, pred_a0 = get_gradcam(model_a0, 'conv6', img)
        axes[row, 2].imshow(cam_a0)
        axes[row, 2].set_title(f'A0 pred: {class_names[pred_a0]}', fontsize=8,
                               color='green' if pred_a0 == true_label else 'red')
    except Exception as e:
        axes[row, 2].text(0.5, 0.5, str(e), ha='center', va='center', fontsize=7, wrap=True)

    # Col 3: A4 GradCAM
    try:
        cam_a4, pred_a4 = get_gradcam(model_a4, 'layer4', img)
        axes[row, 3].imshow(cam_a4)
        axes[row, 3].set_title(f'A4 pred: {class_names[pred_a4]}', fontsize=8,
                               color='green' if pred_a4 == true_label else 'red')
    except Exception as e:
        axes[row, 3].text(0.5, 0.5, str(e), ha='center', va='center', fontsize=7, wrap=True)

for ax in axes.flatten():
    ax.axis('off')

plt.suptitle('GradCAM comparison: CNN scratch (A0) vs ResNet-50 fine-tuned (A4)', fontsize=13)
plt.tight_layout()
plt.savefig('gradcam_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as gradcam_comparison.png')

## 5. Per-class accuracy — worst 10 classes

Identifies the classes that both models struggle with most, for qualitative analysis in the paper.

In [ ]:
from sklearn.metrics import confusion_matrix
from src.data import get_data_loaders
from src.optimization import get_loss
from src.train import one_epoch_test

data_loaders = get_data_loaders(batch_size=64, num_workers=2)
loss_fn = get_loss()

_, _, _, tgt_a4, pred_a4 = one_epoch_test(data_loaders['test'], model_a4, loss_fn)
cm = confusion_matrix(tgt_a4, pred_a4)
per_class_acc = cm.diagonal() / cm.sum(axis=1)

worst_idx = np.argsort(per_class_acc)[:10]
print('10 classes with lowest A4 accuracy:')
print(f'{"Class":<45} {"Accuracy":>8}')
print('-' * 55)
for idx in worst_idx:
    print(f'{class_names[idx]:<45} {100*per_class_acc[idx]:7.1f}%')